# 03 — Main Experiments: the Behavioral Ablation

**The core research question:** do the behavioral indicators (fear / greed / herd) improve
stock-direction prediction *and* trading performance over a quant-only baseline?

We answer it the fair way — same rows, same walk-forward folds, same models; the only thing
that changes is whether `fear/greed/herd` are in the feature matrix. All heavy logic lives in
`src/experiments/ablation.py`; this notebook just runs it and reports.

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))  # so 'import src...' works from notebooks/

import pandas as pd
import matplotlib.pyplot as plt

from src.config import load_config
from src.experiments import run_ablation, run_regime_analysis

pd.set_option('display.float_format', lambda v: f'{v:0.4f}')
config = load_config('../config.yaml')
config['data']['raw_dir'] = '../data/raw'  # reuse the cached snapshot instead of re-downloading
config['data']['tickers']

## Run the ablation

`run_ablation` loads (and caches) each ticker, builds Member A's quant + behavioral features,
labels direction, and evaluates every (model × feature-set) across walk-forward folds. First run
downloads data to `data/raw/`; later runs read the cache.

In [ ]:
results = run_ablation(config)
per_fold, summary, significance = results['per_fold'], results['summary'], results['significance']
print(per_fold.shape)
summary

## Did behavioral features help? (per-fold averages)

Side-by-side accuracy and Sharpe for `quant_only` vs `quant_plus_behavioral`, per model.

In [ ]:
pivot = summary.pivot_table(index=['ticker', 'model'], columns='feature_set',
                            values=['accuracy', 'sharpe'])
pivot

In [ ]:
# Visual: mean accuracy by model and feature set (averaged over tickers).
agg = summary.groupby(['model', 'feature_set'])[['accuracy', 'sharpe']].mean().reset_index()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric in zip(axes, ['accuracy', 'sharpe']):
    agg.pivot(index='model', columns='feature_set', values=metric).plot.bar(ax=ax)
    ax.set_title(f'{metric} by model'); ax.set_xlabel(''); ax.tick_params(axis='x', rotation=0)
    if metric == 'accuracy': ax.axhline(0.5, ls='--', c='grey', lw=1, label='coin flip')
plt.tight_layout(); plt.show()

## Is the gap statistically real?

A paired test across folds (quant+behavioral − quant-only). `mean_diff > 0` favours behavioral;
small `t_pvalue` / `wilcoxon_pvalue` means the gap is unlikely to be fold noise.

In [ ]:
significance

## Which features does the model lean on?

Refit the strongest model on the full quant+behavioral matrix and read its built-in importances —
a quick gut-check on whether fear/greed/herd carry weight. (SHAP, the deeper story, is Member C's
`04_final_results`.)

In [ ]:
from src.experiments.ablation import build_model_frame, feature_columns
from src.data.loader import load_ohlcv
from src.models import build_model

ticker = config['data']['tickers'][0]
raw = load_ohlcv(ticker, config['data']['start_date'], config['data']['end_date'], config['data']['raw_dir'])
frame = build_model_frame(raw, config)
cols = feature_columns(frame, 'quant_plus_behavioral')

model = build_model('random_forest', config).fit(frame[cols], frame['target'])
imp = model.feature_importances().head(15)
ax = imp[::-1].plot.barh(figsize=(7, 6), title=f'{ticker}: top feature importances')
behavioral = {'fear', 'greed', 'herd'}
for label, patch in zip(imp[::-1].index, ax.patches):
    if label in behavioral: patch.set_color('crimson')
plt.tight_layout(); plt.show()
imp

## Stretch: does it help more in turbulent regimes?

Break the behaviour-aware model's out-of-sample performance down by market regime. The thesis is
that fear/greed/herd earn their keep mainly when markets are *not* calm.

In [ ]:
regime_tables = run_regime_analysis(config, model_name='random_forest')
regime_tables[config['data']['tickers'][0]]

## Takeaways (results from the real run: AAPL, MSFT, SPY)

- **Accuracy:** ~**51%** next-day direction (overall mean 0.511; best 0.539 = MSFT/RF/+behavioral). Near chance, as expected for daily equity direction.
- **Behavioral features:** no consistent gain — deltas are tiny (±0.5pp) and flip sign across models. Only **1 of 18** paired tests hit p<0.05, and even that fails Wilcoxon (expected by chance with 18 tests).
- **Robustness:** at a **weekly horizon (horizon=5)** accuracy rises to ~**54%**; behavioral features lean slightly positive for tree models (+1–2pp) but stay non-significant. The null result holds across horizons.
- **Most-used behavioral signal:** under SHAP, **greed** is the single most important feature for MSFT (herd #7, fear #17) — used heavily, but without a generalizable edge.

**Verdict:** behavioral proxies derived from price/volume do not yield a statistically significant edge over technical features — an honest null result, consistent with weak-form market efficiency. See `RESULTS.md` for the full write-up.